In [58]:
import boto3

In [61]:
client = boto3.client("bedrock-runtime", region_name="us-west-2")
# model_id = "deepseek.v3.2"
# model_id = "us.amazon.nova-lite-v1:0"
model_id = "google.gemma-3-4b-it"
# model_id = "anthropic.claude-3-haiku-20240307-v1:0"

def add_user_message(messages, text):
    user_message = {
        "role": "user", 
        "content": [
            {"text": text}
        ]
    }
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {
        "role": "assistant", 
        "content": [
            {"text": text}
        ]    
    }
    messages.append(assistant_message)

# stop_sequences for Antropic models
def chat(messages, system=None, temperature=1.0, stop_sequences=[]):

    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {
            "temperature": temperature,
            # "stopSequences": stop_sequences
        }
    }

    if system:
        params["system"] = [{"text": system}]

    response = client.converse(**params)
    
    return response['output']["message"]["content"][0]["text"]

In [62]:
messages = []

while True:

    message = input("User: ").strip()
    print(f"User: {message}")

    if message == "0":
        break

    if not message:
        continue

    add_user_message(messages, message)

    answer = chat(messages,temperature=1.0)
    
    add_assistant_message(messages, answer)

    print(f"Assistant: {answer}", flush=True)



User: add 2 plus 2
Assistant: 2 + 2 = 4

User: 0


In [50]:
messages = []

add_user_message(messages, "Write 2 sentence description of a fake database")

response = client.converse_stream(messages=messages, modelId=model_id)

text = ""
for event in response["stream"]:
    if "contentBlockDelta" in event:
        chunk = event["contentBlockDelta"]["delta"]["text"]
        print(chunk, end="")
        text += chunk

print(f"\n\nFinal text: {text}")

Okay, here are two sentence descriptions of a fake database:

"This database simulates a collection of customer records for a fictional online bookstore, including details like purchase history, shipping addresses, and contact information."  It's designed for testing and demonstration purposes, and doesn't contain any real user data or valuable information.

Final text: Okay, here are two sentence descriptions of a fake database:

"This database simulates a collection of customer records for a fictional online bookstore, including details like purchase history, shipping addresses, and contact information."  It's designed for testing and demonstration purposes, and doesn't contain any real user data or valuable information.


In [63]:
messages = []

add_user_message(messages, "count from 1 to 10")

chat(messages)

'1, 2, 3, 4, 5, 6, 7, 8, 9, 10\n'